# 地图定位 Agent · 最小可运行 Demo

**目标**：搭一个最小的「自然语言 → 地图工具调用」agent，体会把传统地图 pipeline 包装成 LLM tool 的工程感。

**工具集**（4 个）：
1. `geocode(address)` — 地址 → (lat, lng)
2. `poi_search(keyword, near, radius_m)` — 关键字 + 中心点 → POI 列表
3. `route(origin, destination, mode)` — 起终点 → 路径 / 距离 / ETA
4. `explain_route(route)` — 把结构化路径转成自然语言导航话术

**数据后端**：默认用 *离线 mock*（一个迷你北京样例集），不依赖任何真实 API key，便于教学。
下方注释里给出 *如何替换为高德 / 百度 / Google Maps* 的 hint。

In [ ]:
import os, sys, json, math
sys.path.append(os.path.abspath('../..'))
from utils.llm_client import LLMClient

client = LLMClient(temperature=0.0)
client.model

## 1. Mock 地图数据

几条北京样例数据，覆盖中关村 / 望京 / 三里屯。真实接入时把 `MOCK_*` 替换成 API 调用即可。

In [ ]:
MOCK_GEOCODE = {
    '中关村': (39.9805, 116.3163),
    '望京': (40.0035, 116.4709),
    '三里屯': (39.9367, 116.4570),
    '北京西站': (39.8946, 116.3216),
    '故宫': (39.9163, 116.3972),
}

MOCK_POIS = [
    {'name': '海底捞(中关村店)',  'cat': '川菜', 'lat': 39.9810, 'lng': 116.3170, 'rating': 4.6, 'parking': True},
    {'name': '眉州东坡(望京店)',  'cat': '川菜', 'lat': 40.0030, 'lng': 116.4700, 'rating': 4.5, 'parking': True},
    {'name': '蜀地源(三里屯)',    'cat': '川菜', 'lat': 39.9370, 'lng': 116.4580, 'rating': 4.2, 'parking': False},
    {'name': '南京大牌档(中关村)', 'cat': '江浙', 'lat': 39.9800, 'lng': 116.3180, 'rating': 4.4, 'parking': True},
    {'name': '陈麻婆豆腐(望京)',  'cat': '川菜', 'lat': 40.0040, 'lng': 116.4720, 'rating': 4.7, 'parking': True},
]

def haversine_m(lat1, lng1, lat2, lng2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

## 2. 工具实现

In [ ]:
def tool_geocode(address: str) -> dict:
    """真实接入提示：高德 https://restapi.amap.com/v3/geocode/geo?address=...&key=YOUR_KEY"""
    for k, (lat, lng) in MOCK_GEOCODE.items():
        if k in address:
            return {'address': address, 'lat': lat, 'lng': lng, 'matched': k}
    return {'error': f'未匹配到地址: {address}'}

def tool_poi_search(keyword: str, near: dict, radius_m: int = 2000) -> dict:
    """真实接入提示：高德 /v3/place/around 或 百度 place api"""
    lat0, lng0 = near['lat'], near['lng']
    out = []
    for p in MOCK_POIS:
        d = haversine_m(lat0, lng0, p['lat'], p['lng'])
        if d <= radius_m and (keyword in p['cat'] or keyword in p['name']):
            out.append({**p, 'distance_m': round(d)})
    out.sort(key=lambda x: x['distance_m'])
    return {'count': len(out), 'pois': out}

def tool_route(origin: dict, destination: dict, mode: str = 'walking') -> dict:
    """真实接入提示：高德 /v3/direction/walking 或 driving"""
    d_m = haversine_m(origin['lat'], origin['lng'], destination['lat'], destination['lng'])
    speed = {'walking': 1.3, 'cycling': 4.0, 'driving': 8.0}.get(mode, 1.3)
    eta_s = d_m / speed
    return {
        'mode': mode,
        'distance_m': round(d_m),
        'duration_min': round(eta_s/60, 1),
        'origin': origin,
        'destination': destination,
    }

def tool_explain_route(route: dict) -> dict:
    mode_cn = {'walking':'步行', 'cycling':'骑行', 'driving':'驾车'}.get(route['mode'], route['mode'])
    return {'text': f"{mode_cn}约 {route['distance_m']} 米，预计 {route['duration_min']} 分钟。"}

TOOL_FNS = {
    'geocode': tool_geocode,
    'poi_search': tool_poi_search,
    'route': tool_route,
    'explain_route': tool_explain_route,
}

## 3. 工具 schema（喂给 LLM）

下面是 Anthropic tool use 风格 schema；OpenAI function calling / MCP 同构。

In [ ]:
TOOLS = [
    {
        'name': 'geocode',
        'description': '把中文地址转成经纬度。返回 {lat, lng}。',
        'input_schema': {
            'type': 'object',
            'properties': {'address': {'type': 'string'}},
            'required': ['address'],
        },
    },
    {
        'name': 'poi_search',
        'description': '在指定中心点周围搜索 POI，按类别或关键字过滤，按距离从近到远排序。',
        'input_schema': {
            'type': 'object',
            'properties': {
                'keyword': {'type': 'string', 'description': '类别或关键字，如 川菜 / 海底捞'},
                'near': {'type': 'object', 'properties': {
                    'lat': {'type': 'number'}, 'lng': {'type': 'number'}}, 'required': ['lat','lng']},
                'radius_m': {'type': 'integer', 'default': 2000},
            },
            'required': ['keyword', 'near'],
        },
    },
    {
        'name': 'route',
        'description': '计算从 origin 到 destination 的路径，返回距离与时长。',
        'input_schema': {
            'type': 'object',
            'properties': {
                'origin': {'type': 'object', 'properties': {'lat':{'type':'number'},'lng':{'type':'number'}}, 'required':['lat','lng']},
                'destination': {'type': 'object', 'properties': {'lat':{'type':'number'},'lng':{'type':'number'}}, 'required':['lat','lng']},
                'mode': {'type': 'string', 'enum': ['walking','cycling','driving'], 'default':'walking'},
            },
            'required': ['origin','destination'],
        },
    },
    {
        'name': 'explain_route',
        'description': '把 route 结果翻译成中文导航话术。',
        'input_schema': {
            'type': 'object',
            'properties': {'route': {'type':'object'}},
            'required': ['route'],
        },
    },
]

## 4. Agent 主循环

经典 *while-tool-use* 循环：模型回应里若有 tool_use，就执行并把结果塞回 messages，直到模型输出最终答案。

In [ ]:
SYSTEM = (
    '你是一个地图助理。当用户提出找地点 / 路线 / 周边推荐时，'
    '你必须依次调用工具：\n'
    '1) 用 geocode 把地址转成经纬度；\n'
    '2) 用 poi_search 找候选 POI；\n'
    '3) 选出 Top-1 后用 route 计算到达路径；\n'
    '4) 用 explain_route 生成中文话术；\n'
    '最后用一段中文给出推荐 + 路径解释 + 关键属性（评分 / 是否有停车）。'
    '禁止凭记忆编造经纬度或距离。'
)

def run_agent(user_msg: str, max_steps: int = 8, verbose: bool = True):
    messages = [{'role': 'user', 'content': user_msg}]
    for step in range(max_steps):
        resp = client.chat(messages, system=SYSTEM, tools=TOOLS)
        if verbose:
            print(f'\n--- step {step} stop={resp["stop_reason"]} ---')
        tool_calls = [b for b in resp['content'] if b.get('type') == 'tool_use']
        if not tool_calls:
            text = ''.join(b.get('text','') for b in resp['content'] if b.get('type')=='text')
            return text, messages
        messages.append({'role': 'assistant', 'content': resp['content']})
        tool_results = []
        for tc in tool_calls:
            name, args = tc['name'], tc['input']
            if verbose:
                print(f'[tool] {name}({json.dumps(args, ensure_ascii=False)})')
            try:
                out = TOOL_FNS[name](**args)
            except Exception as e:
                out = {'error': str(e)}
            if verbose:
                print(f'   -> {json.dumps(out, ensure_ascii=False)[:200]}')
            tool_results.append({
                'type': 'tool_result',
                'tool_use_id': tc['id'],
                'content': json.dumps(out, ensure_ascii=False),
            })
        messages.append({'role': 'user', 'content': tool_results})
    return '[max_steps reached]', messages

## 5. 试一个真实需求

> 「我现在在中关村，想找一家 2 公里内、4.5 星以上的川菜，最好有停车，告诉我怎么走。」

In [ ]:
answer, _ = run_agent(
    '我现在在中关村，想找一家 2 公里内、4.5 星以上的川菜，最好有停车，告诉我怎么走。'
)
print('\n========== 最终回复 ==========')
print(answer)

## 6. 试一个跨地点路径需求

In [ ]:
answer, _ = run_agent('从北京西站怎么去故宫，给我步行和驾车两种方案的对比。')
print('\n========== 最终回复 ==========')
print(answer)

## 7. 接入真实地图 API 的要点

把 `tool_geocode` / `tool_poi_search` / `tool_route` 改写为对真实 API 的 `httpx.get(...)` 即可。注意：

- **统一坐标系**：高德返回 GCJ-02，Google 返回 WGS-84，混用会偏移百米级。在工具层做转换，对 LLM 暴露统一坐标。
- **错误兜底**：API 限流 / 无结果 / 模糊匹配，需要在 tool 内 normalize 成统一 `{error: ...}`，让 LLM 能 reflect。
- **超时与重试**：用 `tenacity` 包一层（参考 `utils/llm_client.py` 的写法）。
- **观测**：把每次 tool 调用 log 到 LangSmith / Langfuse，便于事后归因（详见第 12 章）。

### 进一步：包装成 MCP server

把 `TOOL_FNS` 直接挂到一个 `mcp.server.Server` 上（参考第 03 章的 `mcp_demo/server.py`），即可被 Cursor / Claude Desktop / 任何 MCP client 调用。这是把 *部门内部算子* 暴露给整个 agent 生态的最短路径。

### 进一步：训练一个垂直 agent

构造 *(用户口语, 期望 tool 序列)* 数据集，用 GRPO（参考第 09 章）+ verifiable reward（路径距离匹配 / POI 命中率）做小模型 RL，可显著降低线上推理成本。